# CS224N L01 Code Capsule: One-hot vs Dense Word Vectors

**Waypoint**: WP01 — 为什么需要词向量？

**Official anchor**: Slides p19-22; Notes §2.2 (one-hot vectors, Eq.1-2)

## 核心问题

Slides p20 说：*"There is no natural notion of similarity for one-hot vectors!"*

这个 capsule 用代码证明：
1. One-hot 向量之间点积全为 0（正交）→ 无法编码相似性
2. Dense 向量让相似词（hotel/motel）有高余弦相似度

## 运行环境

只需要 `numpy` 和 `matplotlib`（Colab 自带）。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Tiny vocabulary from CS224N slides examples
vocab = ["hotel", "motel", "book", "Seattle", "Seattle_motel"]
vocab_size = len(vocab)
print(f"Vocabulary ({vocab_size} words): {vocab}")

## Part 1: One-hot Encoding

每个词一个位置为 1，其余为 0。维度 = 词汇量。

In [ ]:
one_hot = np.eye(vocab_size)

for i, word in enumerate(vocab):
    print(f"  {word:15s} → {one_hot[i].tolist()}")

print(f"\nDimension = {vocab_size}, Sparsity = {(1 - one_hot.sum() / one_hot.size) * 100:.1f}%")

## Part 2: One-hot 点积 = 0（正交）

关键观察：任意两个不同词的 one-hot 向量点积都是 0。

In [ ]:
dot_matrix_onehot = one_hot @ one_hot.T
norms = np.linalg.norm(one_hot, axis=1)
cos_matrix_onehot = dot_matrix_onehot / np.outer(norms, norms)

print("Cosine similarity (one-hot):")
print(f"{'':15s}", end="")
for w in vocab: print(f"{w:>15s}", end="")
print()
for i, wi in enumerate(vocab):
    print(f"{wi:15s}", end="")
    for j in range(vocab_size):
        print(f"{cos_matrix_onehot[i,j]:15.1f}", end="")
    print()

print(f"\n⚠️  cos(hotel, motel) = {cos_matrix_onehot[0,1]:.1f}  ← 正交！无法区分相似词")

## Part 3: Dense Word Vectors

Slides p21-22: *"We will build a dense vector for each word"*

这里用手构造的 3D 向量演示核心思想：相似词 → 相似向量 → 高余弦相似度。

In [ ]:
dense_vectors = np.array([
    [0.9,  0.1,  0.2],   # hotel
    [0.8,  0.2,  0.1],   # motel (similar to hotel)
    [0.1,  0.9,  0.3],   # book (different)
    [0.2,  0.3,  0.9],   # Seattle (location)
    [0.5,  0.2,  0.6],   # Seattle_motel (mix)
])

dot_dense = dense_vectors @ dense_vectors.T
norms_dense = np.linalg.norm(dense_vectors, axis=1)
cos_dense = dot_dense / np.outer(norms_dense, norms_dense)

print("Cosine similarity (dense):")
print(f"{'':15s}", end="")
for w in vocab: print(f"{w:>15s}", end="")
print()
for i, wi in enumerate(vocab):
    print(f"{wi:15s}", end="")
    for j in range(vocab_size):
        print(f"{cos_dense[i,j]:15.3f}", end="")
    print()

print(f"\n✅ cos(hotel, motel) = {cos_dense[0,1]:.3f}  ← 高相似度！")
print(f"✅ cos(hotel, book)  = {cos_dense[0,2]:.3f}  ← 低相似度")

## Part 4: 可视化对比

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: one-hot heatmap
im1 = axes[0].imshow(cos_matrix_onehot, cmap='RdYlBu_r', vmin=-0.2, vmax=1.0)
axes[0].set_title("One-hot Cosine Similarity\n(All off-diagonal = 0)", fontsize=13, fontweight='bold')
axes[0].set_xticks(range(vocab_size)); axes[0].set_yticks(range(vocab_size))
axes[0].set_xticklabels(vocab, rotation=45, ha='right', fontsize=9)
axes[0].set_yticklabels(vocab, fontsize=9)
for i in range(vocab_size):
    for j in range(vocab_size):
        axes[0].text(j, i, f"{cos_matrix_onehot[i,j]:.1f}", ha="center", va="center", fontsize=10,
                    color="white" if cos_matrix_onehot[i,j] > 0.5 else "black")
plt.colorbar(im1, ax=axes[0], shrink=0.8)

# Right: dense heatmap
im2 = axes[1].imshow(cos_dense, cmap='RdYlBu_r', vmin=-0.2, vmax=1.0)
axes[1].set_title("Dense Vector Cosine Similarity\n(Similar words = high similarity)", fontsize=13, fontweight='bold')
axes[1].set_xticks(range(vocab_size)); axes[1].set_yticks(range(vocab_size))
axes[1].set_xticklabels(vocab, rotation=45, ha='right', fontsize=9)
axes[1].set_yticklabels(vocab, fontsize=9)
for i in range(vocab_size):
    for j in range(vocab_size):
        axes[1].text(j, i, f"{cos_dense[i,j]:.3f}", ha="center", va="center", fontsize=9,
                    color="white" if cos_dense[i,j] > 0.6 else "black")
plt.colorbar(im2, ax=axes[1], shrink=0.8)

plt.suptitle("CS224N L01: One-hot vs Dense Word Vectors", fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
fig2, ax = plt.subplots(figsize=(8, 7))
colors = ['#e74c3c', '#e67e22', '#2ecc71', '#3498db', '#9b59b6']
markers = ['o', 's', '^', 'D', 'v']

for i, word in enumerate(vocab):
    ax.scatter(dense_vectors[i,0], dense_vectors[i,1], c=colors[i], marker=markers[i],
              s=200, zorder=5, edgecolors='black', linewidth=1.5)
    ax.annotate(word, (dense_vectors[i,0], dense_vectors[i,1]),
               textcoords="offset points", xytext=(10,8), fontsize=11, fontweight='bold')

ax.plot([dense_vectors[0,0], dense_vectors[1,0]], [dense_vectors[0,1], dense_vectors[1,1]],
       'g--', lw=2, alpha=0.7, label=f'hotel↔motel (cos={cos_dense[0,1]:.3f})')
ax.plot([dense_vectors[0,0], dense_vectors[2,0]], [dense_vectors[0,1], dense_vectors[2,1]],
       'r--', lw=2, alpha=0.7, label=f'hotel↔book (cos={cos_dense[0,2]:.3f})')

ax.set_xlabel("Dimension 1"); ax.set_ylabel("Dimension 2")
ax.set_title("Dense Word Vectors in 2D: Similar words cluster together", fontsize=13, fontweight='bold')
ax.legend(fontsize=10); ax.grid(True, alpha=0.3); ax.set_aspect('equal')
plt.tight_layout()
plt.show()

## 总结

| 对比项 | One-hot | Dense |
|--------|---------|-------|
| cos(hotel, motel) | 0.0 | 0.987 |
| cos(hotel, book) | 0.0 | 0.271 |
| 维度 | = 词汇量 (50,000+) | 低维 (50-300) |
| 稀疏度 | 100% 零 | 0% 零 |

**Takeaway** (Slides p21-22): Dense vectors 从数据中学习，低维、密集、能编码语义相似性。
这是 Word2Vec 的基础（下一个 capsule）。